# Black-Box Optimisation BBO




## Introduction

Detect likely contamination sources in a two-dimensional area, such as a radiation field, where only proximity yields a non-zero reading. The system uses Bayesian optimisation to tune detection parameters and reliably identify both strong and weak sources.

In [234]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [251]:
# Load the initial data and describe it
f1_input_data = np.load('initial_data/function_1/initial_inputs.npy')
f1_output_data = np.load('initial_data/function_1/initial_outputs.npy')

# First round
f1_input_data = np.concatenate(
    (f1_input_data, [[0.223696, 0.166025], # Week 1
                     [0.971666, 0.994621], # Week 2 
                     [0.580800, 0.403103], # Week 3
                     [0.530137, 0.504108], # Week 4
                     [0.284100, 0.754200], # Week 5
                     [0.879672, 0.190177], # Week 6
                     [0.047162, 0.569522], # Week 7
                     [0.608963, 0.555379], # Week 8
                     [0.568947, 0.768862], # Week 9
                     [0.388166, 0.232975], # Week 10
                     [0.881147, 0.795664], # Week 11
                     [0.339981, 0.323487], # Week 12
                     [0.390337, 0.301715]]), # Week 13
    axis=0
)
f1_output_data = np.append(f1_output_data, 1.2121776220527492e-74) # GP + UCB (iterations = 10) kappa = 8.0
f1_output_data = np.append(f1_output_data, -4.515332508840043e-176) # GP + UCB (iterations = 10) kappa = 20.0
f1_output_data = np.append(f1_output_data, 9.827384409509805e-19) # GP + UCB (iterations = 10) kappa = 12.0  - kappa 10 ????
f1_output_data = np.append(f1_output_data, 3.9242133027962733e-14) # GP + UCB (iterations = 10) kappa = 2.0 - kappa 9 ???
f1_output_data = np.append(f1_output_data, 4.690604757883212e-91) # Neural Networks + UCB (iterations = 10) kappa 2.0
f1_output_data = np.append(f1_output_data, -1.8323318855977174e-178) # GP + EI (iterations = 10) kappa = 0.05
f1_output_data = np.append(f1_output_data, -1.80712027198355e-113) # GP + EI (iterations = 15) kappa = 5.0
f1_output_data = np.append(f1_output_data, -0.0001431460817369449) # GP + EI (iterations = 15) kappa = 10.0
f1_output_data = np.append(f1_output_data, -8.059845136259036e-17) # GP + UCB (iterations = 10) kappa = 9.5
f1_output_data = np.append(f1_output_data, 2.559306584393459e-27) # GP + UCB (iterations = 10) kappa = 5.0
f1_output_data = np.append(f1_output_data, -2.55746372896521e-64) # GP + UCB (iterations = 10) kappa = 6.0
f1_output_data = np.append(f1_output_data, 3.688998813902206e-12) # GP + UCB (iterations = 10) kappa = 0.2
f1_output_data = np.append(f1_output_data, 5.714209335193049e-13) # GP + UCB (iterations = 10) kappa = 0.1

In [252]:
# Read input and out for function 2
f2_input_data = np.load('initial_data/function_2/initial_inputs.npy')
f2_output_data = np.load('initial_data/function_2/initial_outputs.npy')

# First round
f2_input_data = np.concatenate(
    (f2_input_data, [[0.125906, 0.788607], # Week 1
                     [0.985180, 0.251085], # Week 2
                     [0.441830, 0.633407], # Week 3
                     [0.679819, 0.266237], # Week 4
                     [0.657000, 0.212300], # Week 5
                     [0.790743, 0.938552], # Week 6
                     [0.519940, 0.509866], # Week 7
                     [0.703298, 0.466477], # Week 8
                     [0.534528, 0.278183], # Week 9
                     [0.536392, 0.918154], # Week 10
                     [0.011421, 0.198983], # Week 11
                     [0.591362, 0.428469], # Week 12
                     [0.510208, 0.529779]]), # Week 13
    axis=0
)
f2_output_data = np.append(f2_output_data, -0.1739726394059063) # GP + UCB (iterations = 10) kappa = 8.0
f2_output_data = np.append(f2_output_data, -0.01550891697481739) # GP + UCB (iterations = 10) kappa = 20.0
f2_output_data = np.append(f2_output_data, 0.2984716850359006) # GP + UCB (iterations = 10) kappa = 12.0
f2_output_data = np.append(f2_output_data, 0.3520637568002779) # GP + UCB (iterations = 10) kappa = 2.0
f2_output_data = np.append(f2_output_data, 0.36843190917654645) # Neural Networks + UCB (iterations = 10) kappa 2.0
f2_output_data = np.append(f2_output_data, 0.035725127837916065) # GP + EI (iterations = 10) kappa = 0.05
f2_output_data = np.append(f2_output_data, 0.6144819505511563) # GP + EI (iterations = 15) kappa = 5.0
f2_output_data = np.append(f2_output_data, 0.5107611479314056) # GP + EI (iterations = 15) kappa = 10.0
f2_output_data = np.append(f2_output_data, 0.35073554356687237) # GP + EI (iterations = 15) kappa = 4.5
f2_output_data = np.append(f2_output_data, 0.1266592974188294) # GP + EI (iterations = 10) kappa = 3.0
f2_output_data = np.append(f2_output_data, -0.015050624482960325) # GP + EI (iterations = 10) kappa = 4.0
f2_output_data = np.append(f2_output_data, 0.10868422130010974) # GP + EI (iterations = 10) kappa = 0.2  ?? instead of UCB
f2_output_data = np.append(f2_output_data, 0.7770004599039609) # GP + UCB (iterations = 10) kappa = 0.1

In [250]:
# Read input and out for function 2
f3_input_data = np.load('initial_data/function_3/initial_inputs.npy')
f3_output_data = np.load('initial_data/function_3/initial_outputs.npy')

# First round
f3_input_data = np.concatenate(
    (f3_input_data, [[0.370443, 0.100000, 0.100000], [0.100000, 0.900000, 0.900000], [0.559175, 0.100000, 0.618789],
                    [0.418978, 0.452988, 0.516531], [0.907500, 0.234800, 0.979900], [0.733217, 0.902473, 0.900000],
                    [0.056745, 0.644609, 0.812067], [0.401965, 0.444137, 0.317518], [0.460388, 0.697471, 0.592882],
                    [0.710270, 0.560110, 0.653967], [0.303306, 0.000000, 0.411004], [0.801517, 0.472335, 0.709245],
                    [0.437361, 0.461248, 0.586587]]),
    axis=0
)
f3_output_data = np.append(f3_output_data, -0.117375859160532) # GP + UCB (iterations = 10) kappa = 8.0
f3_output_data = np.append(f3_output_data, -0.12336864739293958) # GP + UCB (iterations = 10) kappa = 20.0
f3_output_data = np.append(f3_output_data, -0.11919639248781014) # GP + UCB (iterations = 10) kappa = 12.0
f3_output_data = np.append(f3_output_data, -0.015678917994671488) # GP + UCB (iterations = 10) kappa = 2.0
f3_output_data = np.append(f3_output_data, -0.3698004461896308) # Neural Networks + UCB (iterations = 10) kappa 2.0
f3_output_data = np.append(f3_output_data, -0.09232158961187506) # GP + EI (iterations = 10) kappa = 0.05
f3_output_data = np.append(f3_output_data, -0.06693057594200055) # GP + EI (iterations = 15) kappa = 5.0
f3_output_data = np.append(f3_output_data, -0.06099866510741818) # GP + EI (iterations = 15) kappa = 10.0
f3_output_data = np.append(f3_output_data, -0.04791215400572666) # GP + UCB (iterations = 15) kappa = 0.5
f3_output_data = np.append(f3_output_data, -0.1134040954820501) # GP + UCB (iterations = 10) kappa = 1.0
f3_output_data = np.append(f3_output_data, -0.0740441101108158) # GP + UCB (iterations = 10) kappa = 2.0
f3_output_data = np.append(f3_output_data, -0.11724410952717225) # GP + EI (iterations = 10) kappa = 0.2
f3_output_data = np.append(f3_output_data, -0.041054575325136906) # GP + UCB + PCA (iterations = 10) kappa = 0.1

# print(f3_input_data)
# print("")
# print(f3_output_data)

# Best Next X is: [0.37044368 0.         0.        ]
# Submission: 0.370443-0.100000-0.100000

# Best Next X is: [0. 1. 1.]
# Submission: 0.100000-0.900000-0.900000

# Best Next X is: [0.55917577 0.         0.61878983]  kappa = 2.0
# Submission: 0.559175-0.100000-0.618789

# Best Next X is: [0.41897809 0.45298879 0.51653193] kappa = 1.0 exploitation
# Submission: 0.418978-0.452988-0.516531

# Submission: 0.907500-0.234800-0.979900

# Exploration xi = 0.05
# Submission: 0.733217-0.902473-0.900000

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.056745-0.644609-0.812067

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.401965-0.444137-0.317518

# Best Next X is: [0.46038876 0.69747121 0.59288289] xi = 0.5 GP with UCB - iterations = 15
# Submission: 0.460388-0.697471-0.592882

# Best Next X is: [0.71027034 0.56011058 0.65396794] xi = 1 GP with UCB - iterations = 10
# Submission: 0.710270-0.560110-0.653967

# Best Next X is: [0.30330601 0.         0.41100441] xi = 2 GP with UCB - iterations = 10
# Submission: 0.303306-0.000000-0.411004

# Best Next X is: [0.80151777 0.47233507 0.70924578] xi = 0.2 GP with EI - iterations = 10
# Submission: 0.801517-0.472335-0.709245

# Bext Next X after inverse is : [[0.43736138 0.46124884 0.58658719]] xi=0.1 GP with UCB - iterations = 10 + PCA
# Submission: 0.437361-0.461248-0.586587

In [244]:
# Read input and out for function 2
f4_input_data = np.load('initial_data/function_4/initial_inputs.npy')
f4_output_data = np.load('initial_data/function_4/initial_outputs.npy')

# First round
f4_input_data = np.concatenate(
    (f4_input_data, [[0.900000, 0.391547, 0.100000, 0.100000], [0.100000, 0.900000, 0.900000, 0.900000],
                    [0.405673, 0.390640, 0.346566, 0.428138], [0.407626, 0.408744, 0.294125, 0.437783],
                    [0.342600, 0.558700, 0.611600, 0.502200], [0.685479, 0.753190, 0.168306, 0.737342],
                    [0.577678, 0.500100, 0.677226, 0.175109], [0.172470, 0.073209, 0.372759, 0.989174],
                    [0.424494, 0.339512, 0.445235, 0.454909], [0.679848, 0.950000, 0.950000, 0.950000],
                    [0.332322, 0.568071, 0.035973, 0.674249], [0.588167, 0.570162, 0.563914, 0.107006],
                    [0.587046, 0.596437, 0.489224, 0.572515]]),
    axis=0
)
f4_output_data = np.append(f4_output_data, -19.874502519770903) # GP + UCB (iterations = 10) kappa = 8.0
f4_output_data = np.append(f4_output_data, -34.10576750538245)
f4_output_data = np.append(f4_output_data, 0.46654522988047065)
f4_output_data = np.append(f4_output_data, -0.978378387130991)
f4_output_data = np.append(f4_output_data, -6.075122732992654) # Neural Networks + UCB (iterations = 10) kappa 2.0
f4_output_data = np.append(f4_output_data, -18.37458887015637)
f4_output_data = np.append(f4_output_data, -10.314157202143814)
f4_output_data = np.append(f4_output_data, -21.88994067659473)
f4_output_data = np.append(f4_output_data, -0.3092559607133718)
f4_output_data = np.append(f4_output_data, -40.031478179592405)
f4_output_data = np.append(f4_output_data, -13.22666268459103)
f4_output_data = np.append(f4_output_data, -11.466794676721602)
f4_output_data = np.append(f4_output_data, -8.478525354989117)

# print(f4_input_data)
# print("")
# print(f4_output_data)

# Best Next X is: [1.        0.3915474 0.        0.       ]
# Submission: 0.900000-0.391547-0.100000-0.100000

# Best Next X is: [0. 1. 1. 1.]
# Submission: 0.100000-0.900000-0.900000-0.900000

# Best Next X is: [0.40567321 0.39064003 0.34656646 0.42813842] kappa 2.0
# Submission: 0.405673-0.390640-0.346566-0.428138

# Best Next X is: [0.40762677 0.40874425 0.2941259  0.43778382] kappa 1.0 exploitation
# Submission: 0.407626-0.408744-0.294125-0.437783

#Submission: 0.342600-0.558700-0.611600-0.502200

# Exploration xi = 0.05
# Submission: 0.685479-0.753190-0.168306-0.737342

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.577678-0.500100-0.677226-0.175109

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.172470-0.073209-0.372759-0.989174

# Best Next X is: [0.42449425 0.33951243 0.44523582 0.45490995] xi = 0.5 GP with UCB - iterations = 15
# Submission: 0.424494-0.339512-0.445235-0.454909

# Best Next X is: [0.67984818 1.  1.  1.  ] xi = 4 GP with UCB - iterations = 10
# Submission: 0.679848-0.950000-0.950000-0.950000

# Best Next X is: [0.33232263 0.56807159 0.03597343 0.67424976] xi = 4 GP with EI - iterations = 10
# Submission: 0.332322-0.568071-0.035973-0.674249

# Best Next X is: [0.58816715 0.57016209 0.56391418 0.10700641] xi = 0.2 GP with EI - iterations = 10
# Submission: 0.588167-0.570162-0.563914-0.107006

# Bext Next X after inverse is : [[0.58704658 0.59643701 0.48922414 0.57251596]] xi = 0.1 GP with UCB - iterations = 10 with PCA
# Submission: 0.587046-0.596437-0.489224-0.572515

In [245]:
# Read input and out for function 2
f5_input_data = np.load('initial_data/function_5/initial_inputs.npy')
f5_output_data = np.load('initial_data/function_5/initial_outputs.npy')

# First round
f5_input_data = np.concatenate(
    (f5_input_data, [[0.261146, 0.837257, 0.855685, 0.889760], [0.179332, 0.906546, 0.867288, 0.935252],
                    [0.430812, 0.766167, 0.350393, 0.947247], [0.526768, 0.806704, 0.939242, 0.609982],
                    [0.859900, 0.820800, 0.147800, 0.771800], [0.988643, 0.690716, 0.985391, 0.856409],
                    [0.723047, 0.530092, 0.811697, 0.798753], [0.941048, 0.524466, 0.372639, 0.904600],
                    [0.987432, 0.690041, 0.984631, 0.856124], [0.938284, 0.520819, 0.379362, 0.902919],
                    [0.445315, 0.757693, 0.353784, 0.930553], [0.942718, 0.526670, 0.368575, 0.905615],
                    [0.611960, 0.647068, 0.529179, 0.763135]]),
    axis=0
)
f5_output_data = np.append(f5_output_data, 1003.0955306052545) # GP + UCB (iterations = 10) kappa = 8.0
f5_output_data = np.append(f5_output_data, 1661.4926023721316)
f5_output_data = np.append(f5_output_data, 377.4129493399731)
f5_output_data = np.append(f5_output_data, 642.9609629945041)
f5_output_data = np.append(f5_output_data, 561.8354085559349) # Neural Networks + UCB (iterations = 10) kappa 2.0
f5_output_data = np.append(f5_output_data, 3378.2312831787153)
f5_output_data = np.append(f5_output_data, 384.0959683165886)
f5_output_data = np.append(f5_output_data, 821.4588134967668)
f5_output_data = np.append(f5_output_data, 3350.4545647667915)
f5_output_data = np.append(f5_output_data, 799.5449758520845)
f5_output_data = np.append(f5_output_data, 314.88713830765084)
f5_output_data = np.append(f5_output_data, 835.0079036275938)
f5_output_data = np.append(f5_output_data, 55.941976575026146)

# print(f5_input_data)
# print("")
# print(f5_output_data)

# Best Next X is: [0.2611464  0.8372571  0.85568591 0.88976039]
# Submission: 0.261146-0.837257-0.855685-0.889760

# Best Next X is: [0.1793323  0.90654632 0.867288   0.93525213]
# Submission: 0.179332-0.906546-0.867288-0.935252

# Best Next X is: [0.43081201 0.76616707 0.35039386 0.94724703] kappa 2.0
# Submission: 0.430812-0.766167-0.350393-0.947247

# Best Next X is: [0.52676818 0.80670436 0.93924213 0.60998263] kappa 1.0 exploitation
# Submission: 0.526768-0.806704-0.939242-0.609982

# Submission: 0.859900-0.820800-0.147800-0.771800

# Exploration xi = 0.05 - GP with EI - iter = 15?
# Submission: 0.988643-0.690716-0.985391-0.856409

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.723047-0.530092-0.811697-0.798753

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.941048-0.524466-0.372639-0.904600

# Best Next X is: [0.98743279 0.69004196 0.98463199 0.85612402] xi = 0.5 GP with UCB - iterations = 15
# Submission: 0.987432-0.690041-0.984631-0.856124

# Best Next X is: [0.93828401 0.52081912 0.3793622  0.90291969] xi = 4 GP with UCB - iterations = 10
# Submission: 0.938284-0.520819-0.379362-0.902919

# Best Next X is: [0.44531547 0.75769343 0.35378449 0.93055335] xi = 5 GP with UCB - iterations = 10
# Submission: 0.445315-0.757693-0.353784-0.930553

# Best Next X is: [0.94271857 0.52667029 0.36857554 0.90561599] xi = 0.2 GP with UCB - iterations = 10
# Submission: 0.942718-0.526670-0.368575-0.905615

#Bext Next X after inverse is : [[0.6119604  0.64706833 0.52917956 0.7631359 ]] xi=0.1 GP with UCB - iterations = 10 with PCA
# Submission: 0.611960-0.647068-0.529179-0.763135

In [246]:
# Read input and out for function 2
f6_input_data = np.load('initial_data/function_6/initial_inputs.npy')
f6_output_data = np.load('initial_data/function_6/initial_outputs.npy')

# First round
f6_input_data = np.concatenate(
    (f6_input_data, [[0.100000, 0.100000, 0.100000, 0.100000, 0.100000], # Week 1 
                     [0.900000, 0.900000, 0.100000, 0.100000, 0.900000], # Week 2
                     [0.100000, 0.100000, 0.477694, 0.900000, 0.100000], # Week 3
                     [0.711119, 0.100000, 0.900000, 0.100000, 0.900000], # Week 4
                     [0.120700, 0.480500, 0.335700, 0.771900, 0.529300], # Week 5
                     [0.445702, 0.249091, 0.570676, 0.789337, 0.143917], # Week 6
                     [0.756618, 0.789573, 0.700148, 0.788363, 0.930751], # Week 7
                     [0.224372, 0.323632, 0.710213, 0.752246, 0.958774], # Week 8
                     [0.432407, 0.055250, 0.765700, 0.598069, 0.906864], # Week 9
                     [0.873488, 0.640637, 0.273703, 0.456593, 0.423597], # Week 10
                     [0.663636, 0.734306, 0.985001, 0.291751, 0.164318], # Week 11
                     [0.431880, 0.230272, 0.644232, 0.969621, 0.166391], # Week 12
                     [0.486650, 0.422857, 0.704530, 0.789187, 0.334515]]), # Week 13
    axis=0
)
f6_output_data = np.append(f6_output_data, -1.772464173686114) # GP + UCB (iterations = 10) kappa = 8.0
f6_output_data = np.append(f6_output_data, -3.0710848428181494)
f6_output_data = np.append(f6_output_data, -0.8420881702572393)
f6_output_data = np.append(f6_output_data, -2.25609789259345)
f6_output_data = np.append(f6_output_data, -1.2030646453198108) # Neural Networks + UCB (iterations = 10) kappa 2.0
f6_output_data = np.append(f6_output_data, -0.27914755272875114)
f6_output_data = np.append(f6_output_data, -1.6262062830456927)
f6_output_data = np.append(f6_output_data, -1.2556062668293657)
f6_output_data = np.append(f6_output_data, -1.4916821089090682)
f6_output_data = np.append(f6_output_data, -1.8078362346120846)
f6_output_data = np.append(f6_output_data, -1.4564902307553127)
f6_output_data = np.append(f6_output_data, -0.4295074199665059)
f6_output_data = np.append(f6_output_data, -0.36033278793561724)

# print(f6_input_data)
# print("")
# print(f6_output_data)

# Best Next X is: [0.1  0.1  0.1  0.1  0.1 ]
# Submission: 0.100000-0.100000-0.100000-0.100000-0.100000 

# Best Next X is: [1. 1. 0. 0. 1.]
# Submission: 0.900000-0.900000-0.100000-0.100000-0.900000

# Best Next X is: [0.         0.         0.47769496 1.         0.        ] kappa = 4.0 
# Submission: 0.100000-0.100000-0.477694-0.900000-0.100000

# Best Next X is: [0.71111973 0.         1.         0.         1.        ] kappa = 2.0 exploitation
# Submission: 0.711119-0.100000-0.900000-0.100000-0.900000

# Submission: 0.120700-0.480500-0.335700-0.771900-0.529300

# Exploration xi = 0.05
# Submission: 0.445702-0.249091-0.570676-0.789337-0.143917

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.756618-0.789573-0.700148-0.788363-0.930751

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.224372-0.323632-0.710213-0.752246-0.958774

# Best Next X is: [0.43240749 0.05525024 0.76570078 0.59806901 0.90686457] GP + EI xi=0.5 iter = 15
# Submission: 0.432407-0.055250-0.765700-0.598069-0.906864

# Best Next X is: [0.87348815 0.64063719 0.27370341 0.45659391 0.4235978 ] GP + EI xi=3 iter = 10
# Submission: 0.873488-0.640637-0.273703-0.456593-0.423597 

# Best Next X is: [0.66363629 0.73430657 0.98500134 0.29175123 0.16431885] GP + EI xi=4 iter = 10
# Submission: 0.663636-0.734306-0.985001-0.291751-0.164318

# Best Next X is: [0.43188076 0.2302724  0.64423278 0.9696219  0.16639171] GP + UCB xi=0.2 iter = 10
# Submission: 0.431880-0.230272-0.644232-0.969621-0.166391

# Bext Next X after inverse is : [[0.4866504  0.4228571  0.70453041 0.78918726 0.3345156 ]] GP +UCB xi =0.1 iter =10 with PCA
# Submission: 0.486650-0.422857-0.704530-0.789187-0.334515

In [247]:
# Read input and out for function 2
f7_input_data = np.load('initial_data/function_7/initial_inputs.npy')
f7_output_data = np.load('initial_data/function_7/initial_outputs.npy')

# First round
f7_input_data = np.concatenate(
    (f7_input_data, [[0.158904, 0.131800, 0.383454, 0.110947, 0.292992, 0.900000], # Week 1
                     [0.900000, 0.900000, 0.900000, 0.900000, 0.900000, 0.900000], # Week 2
                    [0.100000, 0.439365, 0.535234, 0.100000, 0.367470, 0.975232], # Week 3
                    [0.100000, 0.199729, 0.201684, 0.103729, 0.310943, 0.748939], # Week 4
                    [0.770200, 0.032400, 0.948200, 0.525000, 0.845400, 0.393100], # Week 5
                    [0.958389, 0.843644, 0.711162, 0.307427, 0.256341, 0.429319], # Week 6
                    [0.394426, 0.606437, 0.005884, 0.800916, 0.534595, 0.337790], # Week 7
                    [0.023624, 0.028917, 0.684768, 0.017858, 0.829508, 0.112882], # Week 8
                    [0.100000, 0.198247, 0.173095, 0.082758, 0.313939, 0.763621], # Week 9
                    [0.101717, 0.217655, 0.337649, 0.203732, 0.293678, 0.673045], # Week 10
                    [0.097177, 0.221951, 0.410126, 0.256124, 0.300504, 0.618146], # Week 11
                    [0.083137, 0.214957, 0.413178, 0.260460, 0.309334, 0.607274], # Week 12
                    [0.455074, 0.378790, 0.414111, 0.453655, 0.465969, 0.523820]]), # Week 13
    axis=0
)
f7_output_data = np.append(f7_output_data, 1.551275860667744) # GP + UCB (iterations = 10) kappa = 8.0
f7_output_data = np.append(f7_output_data, 0.0005215039140009245)
f7_output_data = np.append(f7_output_data, 0.9143578861897479)
f7_output_data = np.append(f7_output_data, 1.8757352131149276)
f7_output_data = np.append(f7_output_data, 0.001978776109030435) # Neural Networks + UCB (iterations = 10) kappa 2.0
f7_output_data = np.append(f7_output_data, 0.05806318383973897)
f7_output_data = np.append(f7_output_data, 0.2926005954433965)
f7_output_data = np.append(f7_output_data, 0.10481124129157648)
f7_output_data = np.append(f7_output_data, 1.6525247601700708)
f7_output_data = np.append(f7_output_data, 2.8168922468694917)
f7_output_data = np.append(f7_output_data, 3.029156813606421)
f7_output_data = np.append(f7_output_data, 2.992904995633214)
f7_output_data = np.append(f7_output_data, 0.9856429540383217)

# print(f7_input_data)
# print("")
# print(f7_output_data)

# Best Next X is: [0.15890439 0.13180032 0.38345458 0.11094758 0.29299283 1.        ] UCB 8.0 lower bound
# Submission: 0.158904-0.131800-0.383454-0.110947-0.292992-0.900000

# Best Next X is: [1. 1. 1. 1. 1. 1.]
# Submission: 0.900000-0.900000-0.900000-0.900000-0.900000-0.900000

#Best Next X is: [0.         0.43936567 0.53523459 0.         0.36747006 0.97523208] kappa = 4.0
# Submission: 0.100000-0.439365-0.535234-0.100000-0.367470-0.975232

# Best Next X is: [0.         0.19972912 0.20168429 0.10372903 0.31094381 0.74893963] kappa = 2.0 exploitation
# Submission: 0.100000-0.199729-0.201684-0.103729-0.310943-0.748939

# Submission: 0.770200-0.032400-0.948200-0.525000-0.845400-0.393100

# Exploration xi = 0.05
# Submission: 0.958389-0.843644-0.711162-0.307427-0.256341-0.429319

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.394426-0.606437-0.005884-0.800916-0.534595-0.337790

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.023624-0.028917-0.684768-0.017858-0.829508-0.112882

# Best Next X is: [0.         0.19824735 0.17309552 0.08275881 0.31393931 0.76362193] xi = 1.0 GP with UCB - iterations = 15
# Submission: 0.100000-0.198247-0.173095-0.082758-0.313939-0.763621

# Best Next X is: [0.1017177  0.2176553  0.33764912 0.20373241 0.29367852 0.67304558] xi = 0.5 GP with UCB - iterations = 10
# Submission: 0.101717-0.217655-0.337649-0.203732-0.293678-0.673045

# Best Next X is: [0.09717797 0.22195104 0.41012667 0.25612456 0.30050477 0.61814615] xi = 0.5 GP with UCB - iterations = 10
# Submission: 0.097177-0.221951-0.410126-0.256124-0.300504-0.618146

# Best Next X is: [0.08313786 0.21495797 0.4131781  0.26046091 0.30933403 0.60727456] xi = 0.2 GP with UCB - iterations = 10
# Submission: 0.083137-0.214957-0.413178-0.260460-0.309334-0.607274

# Bext Next X after inverse is : [[0.45507441 0.37879007 0.41411188 0.45365547 0.46596959 0.52382049]] xi=0.1 GP with UCB - iter =10
# Submission: 0.455074-0.378790-0.414111-0.453655-0.465969-0.523820

In [248]:
# Read input and out for function 2
f8_input_data = np.load('initial_data/function_8/initial_inputs.npy')
f8_output_data = np.load('initial_data/function_8/initial_outputs.npy')

# First round
f8_input_data = np.concatenate(
    (f8_input_data, [[0.100000, 0.100000, 0.100000, 0.100000, 0.900000, 0.100000, 0.100000, 0.900000], # Week 1
                     [0.100000, 0.900000, 0.100000, 0.900000, 0.900000, 0.900000, 0.100000, 0.900000], # Week 2
                    [0.100000, 0.058521, 0.118593, 0.021611, 0.900000, 0.748495, 0.157896, 0.576730], # Week 3
                    [0.150061, 0.263393, 0.100000, 0.078784, 0.681402, 0.155068, 0.120381, 0.071284], # Week 4
                    [0.197900, 0.682800, 0.352600, 0.859300, 0.417900, 0.660200, 0.907500, 0.559300], # Week 5
                    [0.297223, 0.725239, 0.450101, 0.260713, 0.017052, 0.457917, 0.171118, 0.506765], # Week 6
                    [0.919172, 0.231566, 0.578186, 0.028455, 0.467396, 0.882039, 0.141356, 0.363101], # Week 7
                    [0.281217, 0.981014, 0.198875, 0.969273, 0.694379, 0.029533, 0.494993, 0.053073], # Week 8
                    [0.044472, 0.358709, 0.000000, 0.256740, 0.542412, 0.569371, 0.102869, 0.580490], # Week 9
                    [0.091430, 0.153888, 0.173259, 0.087793, 0.737778, 0.470758, 0.223349, 0.647339], # Week 10
                    [0.128093, 0.183460, 0.124892, 0.170784, 0.809712, 0.509660, 0.222141, 0.613314], # Week 11
                    [0.134207, 0.173346, 0.133126, 0.150276, 0.757313, 0.500828, 0.208864, 0.572960], # Week 12
                    [0.686136, 0.467986, 0.557699, 0.582667, 0.508812, 0.525244, 0.631114, 0.642329]]), # Week 13
    axis=0
)
f8_output_data = np.append(f8_output_data, 9.7983) # GP + UCB (iterations = 10) kappa = 8.0
f8_output_data = np.append(f8_output_data, 8.6783)
f8_output_data = np.append(f8_output_data, 9.904408090344)
f8_output_data = np.append(f8_output_data, 9.8077148428394)
f8_output_data = np.append(f8_output_data, 7.945254176000001) # Neural Networks + UCB (iterations = 10) kappa 2.0
f8_output_data = np.append(f8_output_data, 8.9608425698375)
f8_output_data = np.append(f8_output_data, 7.8201193386419)
f8_output_data = np.append(f8_output_data, 8.127425233881599)
f8_output_data = np.append(f8_output_data, 9.831285357306)
f8_output_data = np.append(f8_output_data, 9.9862489266639)
f8_output_data = np.append(f8_output_data, 9.9956530943604)
f8_output_data = np.append(f8_output_data, 9.9959433020615)
f8_output_data = np.append(f8_output_data, 8.0608638244959)

# print(f8_input_data)
# print("")
# print(f8_output_data)

#Best Next X is: [0.1 0.1 0.1 0.1 1. 0.1 0.1 1.]  UCB 8.0 lower bound
# Submission: 0.100000-0.100000-0.100000-0.100000-0.900000-0.100000-0.100000-0.900000

# Best Next X is: [0. 1. 0. 1. 1. 1. 0. 1.]
# Submission: 0.100000-0.900000-0.100000-0.900000-0.900000-0.900000-0.100000-0.900000

# Best Next X is: [0.   0.05852111 0.1185931  0.02161164 1.   0.74849565 0.15789648 0.57673056] kappa = 3.0
# Submission: 0.100000-0.058521-0.118593-0.021611-0.900000-0.748495-0.157896-0.576730

# Best Next X is: [0.15006142 0.26339307 0.  0.07878424 0.68140243 0.15506866 0.12038117 0.07128475] kappa = 2.0 exploitation
# Submission: 0.150061-0.263393-0.100000-0.078784-0.681402-0.155068-0.120381-0.071284

# Submission: 0.197900-0.682800-0.352600-0.859300-0.417900-0.660200-0.907500-0.559300

# Exploration xi = 0.05
# Submission: 0.297223-0.725239-0.450101-0.260713-0.017052-0.457917-0.171118-0.506765

# Exploration xi = 5 - Gaussian Process with EI - iterations = 15
# Submission: 0.919172-0.231566-0.578186-0.028455-0.467396-0.882039-0.141356-0.363101

# Exploration xi = 10 - Gaussian Process with EI - iterations = 15
# Submission: 0.281217-0.981014-0.198875-0.969273-0.694379-0.029533-0.494993-0.053073

# Best Next X is: [0.04447244 0.35870941 0. 0.25674018 0.54241252 0.56937197 0.10286919 0.58049012] xi = 2.5 GP with UCB - iterations = 15
# Submission: 0.044472-0.358709-0.000000-0.256740-0.542412-0.569371-0.102869-0.580490

# Best Next X is:[0.0914307  0.15388878 0.17325998 0.08779346 0.73777814 0.47075854 0.22334918 0.64733978] xi = 0.5 GP with UCB - iter=10
# Submission: 0.091430-0.153888-0.173259-0.087793-0.737778-0.470758-0.223349-0.647339

# Best Next X is: [0.12809348 0.18346099 0.12489269 0.17078474 0.80971299 0.5096602 0.22214161 0.61331462] xi = 0.5 GP with UCB - iter=10
# Submission: 0.128093-0.183460-0.124892-0.170784-0.809712-0.509660-0.222141-0.613314

# Best Next X is: [0.13420712 0.17334634 0.13312659 0.15027613 0.75731353 0.5008288 0.20886415 0.57296035] xi = 0.2 GP with UCB - iter=10
# Submission: 0.134207-0.173346-0.133126-0.150276-0.757313-0.500828-0.208864-0.572960

#Bext Next X after inverse is : [[0.6861369  0.46798638 0.55769959 0.58266731 0.50881276 0.52524491 0.63111417 0.64232906]]
# xi=0.1 GP with UCB - iter = 10 with PCA
# Submission: 0.686136-0.467986-0.557699-0.582667-0.508812-0.525244-0.631114-0.642329

In [199]:
# Acquisition function - UCB
def apply_pca(X, y):
    """
    Apply PCA.
    
    Parameters:
    -----------
    X : input value dimensions
    y : output values for the dimensions
    """
    # scaler = StandardScaler()
    # X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)  # reduced from no dimensions to 2
    X_reduced = pca.fit_transform(X)
    
    return X_reduced

# print(len(f3_input_data))
# print(len(f3_output_data))
# print(f3_input_data)
# result = apply_pca(f3_input_data, f3_output_data)
# print(result)

In [200]:
# Acquisition function - UCB
def upper_confidence_bound(mu, sigma, kappa):
    """
    Upper Confidence Bound (UCB) acquisition function.
    
    UCB = mean + kappa * std
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation
    kappa : exploration parameter (higher = more exploration)
    """
    return mu + kappa * sigma


In [201]:

# In your EI definition, larger xi makes the method more exploitative, and smaller xi makes it more exploratory.
def expected_improvement(mu, sigma, y_best, xi=0.01):
    """
    Expected Improvement (EI) acquisition function.
    
    EI = E[max(f(x) - f(x_best), 0)]
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation  
    y_best : best observed value so far
    xi : exploration parameter
    """
    with np.errstate(divide='warn'):
        improvement = mu - y_best - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei
    

In [216]:
def bayesian_optimization_nd(X_samples, y_samples, n_dims, n_iterations=50, 
                            n_initial=10):
    """
    Bayesian Optimization for n-dimensional functions.
    Assumes bounds [0, 1] for all dimensions.
    """
    bounds = [(0, 1) for _ in range(n_dims)]
    
    best_values = [y_samples.max()]

    # Scale outputs
    # y_samples = (y_samples - y_samples.mean()) / y_samples.std()

    X_next_best = []
    
    print(f"Starting {n_dims}D optimization with {n_initial} initial samples...")
    print(f"Initial best y: {y_samples.max():.20f}\n")
    
    for iteration in range(n_iterations):
        # Fit GP
        kernel = ConstantKernel(1.0, (1e-8, 1e3)) * RBF(0.3, (1e-4, 10))
        gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                     n_restarts_optimizer=5)
        gp.fit(X_samples, y_samples)
        
        # Optimize acquisition function
        def acq_objective(X):
            X = X.reshape(1, -1)
            mu, sigma = gp.predict(X, return_std=True)
            
           # Kappa initially 8.0 , after 20.0, after 12.00 (explore the middle), 2.0 default or used for NN
            acq_val = upper_confidence_bound(mu, sigma, kappa=0.1)

            # Exploration xi = 0.05 Week 6
            # Exploitation xi = 5 Week 7
            # Explitation xi = 10 Week 8
            # acq_val = expected_improvement(mu, sigma, best_values[0], xi=0.2)
            
            return -acq_val
        
        # Multi-start optimization
        best_acq = np.inf
        X_next = None

        for _ in range(20):  # More starts for higher dimensions
            x0 = np.random.uniform(0, 1, n_dims)
            result = minimize(acq_objective, x0, bounds=bounds, method='L-BFGS-B')
            if result.fun < best_acq:
                best_acq = result.fun
                X_next = result.x
                X_next_best.append(X_next)
                print("Best Next X is: " + str(X_next))

    print(len(X_next_best))
    return X_next_best, y_samples, best_values


# Run 8D optimization
X_samples_8d, y_samples_8d, best_values_8d = bayesian_optimization_nd(
    f8_input_data, f8_output_data, 
    n_dims=8,
    n_iterations=10,
)


Starting 8D optimization with 10 initial samples...
Initial best y: 9.99594330206149983553

Best Next X is: [0.10867587 0.18650662 0.1458669  0.15309773 0.77362802 0.53233095
 0.22341549 0.68468575]
Best Next X is: [0.10869178 0.1865063  0.14586633 0.15309679 0.77363739 0.53234236
 0.22341101 0.68464717]
Best Next X is: [0.10869638 0.18649563 0.14586566 0.15310049 0.77361743 0.53232244
 0.22341963 0.68463768]
Best Next X is: [0.10868962 0.18650767 0.14586665 0.15310672 0.77362125 0.53234948
 0.22341952 0.68465989]
Best Next X is: [0.10868089 0.18650402 0.14586839 0.15311991 0.7736235  0.53237044
 0.22341819 0.68465283]
Best Next X is: [0.10868728 0.18649094 0.14586679 0.15309741 0.77363429 0.532351
 0.22341373 0.68468487]
Best Next X is: [0.10869317 0.18649333 0.14586601 0.15309982 0.77361682 0.53232796
 0.2234234  0.6846555 ]
Best Next X is: [0.1086915  0.18649828 0.14586977 0.15309589 0.77362627 0.53234654
 0.223416   0.68463691]
Best Next X is: [0.10868878 0.18649358 0.14586548 0.15

In [179]:
def apply_pca(X, y):
    """
    Apply PCA.
    
    Parameters:
    -----------
    X : input value dimensions
    y : output values for the dimensions
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)  # reduced from no dimensions to 2
    X_reduced = pca.fit_transform(X_scaled)

    X_samples, y_samples, best_values_8d = bayesian_optimization_nd(X_reduced, y, n_dims=2, n_iterations=10)
    for X_sample in X_samples:
        X_reshaped = X_sample.reshape(1, -1)
        # Step 1: inverse PCA
        x_next_scaled = pca.inverse_transform(X_reshaped)

        # Step 2: inverse scaling
        x_next_original = scaler.inverse_transform(x_next_scaled)
        print("Bext Next X after inverse is : " + str(x_next_original))
    
    return X_reduced

apply_pca(f8_input_data, f8_output_data)

Starting 2D optimization with 10 initial samples...
Initial best y: 9.99594330206149983553

Best Next X is: [0.79101049 0.94432668]
Best Next X is: [0.55898431 0.        ]
Best Next X is: [0.55898441 0.        ]
Best Next X is: [0.55898446 0.        ]
Best Next X is: [0.79101089 0.94432658]
Best Next X is: [0.79101082 0.94432659]
Best Next X is: [0.55898446 0.        ]
Best Next X is: [0.55898446 0.        ]
Best Next X is: [0.55898445 0.        ]
Best Next X is: [0.55898444 0.        ]
Best Next X is: [0.55898445 0.        ]
Best Next X is: [0.55898448 0.        ]
Best Next X is: [0.79101058 0.94432661]
Best Next X is: [0.55898443 0.        ]
Best Next X is: [0.79101098 0.94432668]
Best Next X is: [0.55898447 0.        ]
Best Next X is: [0.55898445 0.        ]
Best Next X is: [0.79101061 0.94432665]
Best Next X is: [0.55898449 0.        ]
Best Next X is: [0.55898444 0.        ]
Best Next X is: [0.7910107  0.94432662]
Best Next X is: [0.55898453 0.        ]
Best Next X is: [0.55898442 

array([[ 0.58670184,  0.56761347],
       [ 1.03387214, -1.49721917],
       [ 0.22310567, -2.25061639],
       [ 1.0188098 , -1.5464351 ],
       [ 0.49859305,  0.21347244],
       [-0.47849139,  1.59504267],
       [ 0.4058579 ,  2.35512304],
       [ 1.56168127, -1.18410395],
       [ 1.18112147, -1.27027441],
       [ 2.00703869, -0.7194028 ],
       [ 0.71801552,  0.20753999],
       [ 1.84058966, -0.02579682],
       [-0.25534295, -2.20998397],
       [ 1.12649786,  1.57477541],
       [-2.51260721,  0.60472851],
       [ 0.3430051 ,  0.76112452],
       [ 1.65361712, -1.34986168],
       [ 1.21576332, -0.0912188 ],
       [ 0.85784833, -0.77813055],
       [-0.22059035, -1.6063765 ],
       [ 1.29121299,  1.90503187],
       [ 1.68054738,  0.27350892],
       [-0.93280837,  0.67613369],
       [-0.2641209 , -0.18679419],
       [ 0.92794219,  0.50878854],
       [-0.63779075, -0.16318793],
       [-0.71921134, -0.48561192],
       [ 0.45620691,  2.38687458],
       [-0.0285516 ,

In [80]:
# Neural Networks using torch with gradient descent 
# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SurrogateFunctionNeuralNetworks(nn.Module):
    def __init__(self, input_dimensions):
        super().__init__()
        # Define layers
        self.fc1 = nn.Linear(input_dimensions, 64)  # input -> hidden 1
        self.fc2 = nn.Linear(64, 32)          # hidden 1 -> hidden 2
        self.fc3 = nn.Linear(32, 1)           # hidden 2 -> output
        self.dropout = nn.Dropout(p=0.3)      # regularisation

    def forward(self, x):
        x = F.relu(self.fc1(x))   # first hidden layer
        x = self.dropout(x)       # drop 30 % of neurons during training
        x = F.relu(self.fc2(x))   # second hidden layer
        x = self.dropout(x)
        x = torch.sigmoid(self.fc3(x))  # output probability in [0, 1]
        return x




In [81]:
# Optimisation aquisition function
# def optimised_acquisition(dimensions):
#     model  = SurrogateFunctionNeuralNetworks(dimensions)
#     optimizer = optim.Adam(model.parameters(), lr=1e-3)

#     for epoch in range(2000):
#         optimizer.zero_grad()
#         pred = model(X)
#         loss = F.mse_loss(pred.squeeze(), y)
#         loss.backward()
#         optimizer.step()

In [82]:
# Epoch looop and get the suggested inputs
def predict(model, x, n_samples=5):
    # model.train()  # important: keep dropout active
    preds = torch.stack([model(x) for _ in range(n_samples)])
    mean = preds.mean(dim=0)
    std = preds.std(dim=0)
    return mean, std

def get_next_best_points(inputs, outputs, dimensions, iterations):
    X_training = torch.tensor(inputs, dtype=torch.float32)
    y_training = torch.tensor(outputs, dtype=torch.float32)

    model  = SurrogateFunctionNeuralNetworks(dimensions)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    bounds = [(0, 1) for _ in range(dimensions)]
    
    best_values = [y_training.max()]

    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        pred = model(X_training)
        loss = F.mse_loss(pred.squeeze(), y_training)
        loss.backward()
        optimizer.step()

    # Neural Network kappa = 2.0 for all functions
    def acq_objective(x):
        # x = torch.rand(1, dimensions, requires_grad=True)
        # print(x)
        mu, sigma = predict(model, x)
            
        acq_val = upper_confidence_bound(mu, sigma, kappa=2.0)
            
        return -acq_val

    x = torch.rand(1, dimensions, requires_grad=True)
    optimizer = torch.optim.Adam([x], lr=1e-3)

    for _ in range(20):
        optimizer.zero_grad()
        ei = acq_objective(x)
        loss = -ei  # maximize EI
        loss.backward()
        optimizer.step()
        x.data.clamp_(0.0, 1.0)  # enforce bounds

    # print(x.detach())
    return x.detach()
        
    # Multi-start optimization
    # best_acq = np.inf
    # X_next = None
        
    # for _ in range(20):  # More starts for higher dimensions
    #     x0 = np.random.uniform(0, 1, dimensions)
    #     print(x0)
    #     result = minimize(acq_objective, x0, bounds=bounds, method='L-BFGS-B')
    #     result = result.detach().numpy()
    #     print(result)
    #     if result.fun < best_acq:
    #         best_acq = result.fun
    #         X_next = result.x
    #         print("Best Next X is: " + str(X_next))
    
    # return  best_values


_ = get_next_best_points(
    f1_input_data, f1_output_data, 
    dimensions=2,
    iterations=10,
)

for iteration in range(20):
    x_next = get_next_best_points(f8_input_data, f8_output_data, dimensions=8,iterations=10)
    print(x_next)

    

tensor([[0.1789, 0.6251, 0.7307, 0.3930, 0.4769, 0.2103, 0.4704, 0.3965]])
tensor([[0.9306, 0.5880, 0.9093, 0.5118, 0.7238, 0.1823, 0.2379, 0.5551]])
tensor([[0.3659, 0.9431, 0.2389, 0.5275, 0.4049, 0.0634, 0.5967, 0.0450]])
tensor([[0.9610, 0.8984, 0.9737, 0.2469, 0.9574, 0.1226, 0.3884, 0.5957]])
tensor([[0.0898, 0.5247, 0.6076, 0.0181, 0.6257, 0.3507, 1.0000, 0.3061]])
tensor([[0.6518, 0.5775, 0.7808, 0.4551, 0.6129, 0.1738, 0.2324, 0.9471]])
tensor([[0.1617, 0.6605, 0.8678, 0.7292, 0.0624, 0.3388, 0.0303, 0.7854]])
tensor([[0.6141, 0.1683, 0.7234, 0.0728, 0.1283, 0.1268, 0.5571, 0.8325]])
tensor([[0.3458, 0.5636, 0.1147, 0.2622, 0.0101, 0.1123, 0.9888, 0.5083]])
tensor([[0.6882, 0.4858, 0.5418, 0.9916, 0.0991, 0.2607, 0.2353, 0.1807]])
tensor([[0.6765, 0.3195, 0.2891, 0.2661, 0.5087, 0.8612, 0.7039, 0.6743]])
tensor([[0.4158, 0.3614, 0.7817, 0.5824, 0.4677, 0.9624, 0.6212, 0.9362]])
tensor([[0.1930, 0.8916, 0.8539, 0.9804, 0.2995, 0.5827, 0.2333, 0.5384]])
tensor([[0.5673, 0.2270, 